# Modulo 9 · Lezione 2 — Pulizia e Preparazione dei Dati

**Progetto Finale: Sistema Intelligente di Predizione ed Esposizione di Modelli**

---

## Cosa vedremo oggi?

| # | Argomento |
|---|------|
| 1 | Audit della qualità dei dati |
| 2 | Trattamento di duplicati e valori nulli |
| 3 | Outlier: rilevamento e decisione |
| 4 | Encoding e trasformazioni |
| 5 | Feature engineering e salvataggio finale |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats

# Caricare i dati grezzi della lezione precedente
df_raw = pd.read_parquet('data/raw/vinos_combinados_raw.parquet')
print(f"Dataset grezzo caricato: {df_raw.shape}")
df_raw.head(3)

---
## 1. Audit della qualità dei dati

In [ ]:
def auditoria_calidad(df):
    """Report completo di qualità di un DataFrame."""
    print(f"{'AUDIT DI QUALITÀ':=^60}")
    print(f"  Righe: {df.shape[0]:,}  |  Colonne: {df.shape[1]}")
    print(f"  Memoria: {df.memory_usage(deep=True).sum()/1024:.0f} KB")

    print(f"\n{'Colonna':30} {'Tipo':10} {'Nulli':>6} {'%':>6} {'Unici':>8} {'Min':>10} {'Max':>10}")
    print('-' * 85)
    for col in df.columns:
        nulos   = df[col].isna().sum()
        pct     = nulos / len(df) * 100
        unicos  = df[col].nunique()
        if pd.api.types.is_numeric_dtype(df[col]):
            mn, mx = df[col].min(), df[col].max()
        else:
            mn, mx = '-', '-'
        print(f"  {col:28} {str(df[col].dtype):10} {nulos:6} {pct:6.1f}% {unicos:8} {str(mn):>10} {str(mx):>10}")

    print(f"\n  Duplicati esatti: {df.duplicated().sum()}")
    print(f"  Correlazione massima tra feature (escl. diagonale):")
    num = df.select_dtypes('number')
    corr = num.corr().abs()
    np.fill_diagonal(corr.values, 0)
    idx = np.unravel_index(corr.values.argmax(), corr.shape)
    print(f"    {corr.columns[idx[0]]} ↔ {corr.columns[idx[1]]}: {corr.values.max():.3f}")

auditoria_calidad(df_raw)

---
## 2. Trattamento di duplicati e valori nulli

In [ ]:
df = df_raw.copy()
print(f"Dataset originale: {df.shape}")

# --- Duplicati ---
n_dup = df.duplicated().sum()
print(f"\nDuplicati trovati: {n_dup}")
if n_dup > 0:
    # Mostrare un esempio
    print("Esempio di duplicati:")
    print(df[df.duplicated(keep=False)].head(4))
    # Nei dati sui vini, i duplicati sono valori FISICAMENTE POSSIBILI
    # (due vini possono avere esattamente le stesse caratteristiche)
    # Decisione: eliminarli per evitare data leakage nella validazione incrociata
    df = df.drop_duplicates().reset_index(drop=True)
    print(f"\nDopo l'eliminazione dei duplicati: {df.shape} ({n_dup} righe eliminate)")

# --- Valori nulli ---
n_nulos = df.isna().sum().sum()
print(f"\nValori nulli totali: {n_nulos}")
if n_nulos > 0:
    print(df.isna().sum()[df.isna().sum() > 0])
else:
    print("✓ Nessun valore nullo — l'imputazione non è necessaria")

# Simulare valori nulli per la demo (in un progetto reale verrebbero dai dati)
print("\nSimulazione di valori nulli per dimostrare le strategie di imputazione:")
df_con_nulos = df.copy()
np.random.seed(42)
for col in ['pH', 'alcohol', 'sulphates']:
    idx_nulos = np.random.choice(len(df_con_nulos), size=20, replace=False)
    df_con_nulos.loc[idx_nulos, col] = np.nan

print(f"  Nulli aggiunti: {df_con_nulos.isna().sum().sum()}")

# Strategie di imputazione
print("\nStrategie di imputazione:")
for estrategia, func in [
    ('Mediana',  lambda s: s.fillna(s.median())),
    ('Media',    lambda s: s.fillna(s.mean())),
    ('Moda',     lambda s: s.fillna(s.mode()[0])),
]:
    df_temp = df_con_nulos.copy()
    for col in ['pH', 'alcohol', 'sulphates']:
        df_temp[col] = func(df_temp[col])
    print(f"  {estrategia}: {df_temp.isna().sum().sum()} nulli rimanenti")

# Nel nostro dataset originale non ci sono nulli → continuiamo senza imputare
print(f"\nDataset finale senza duplicati né nulli: {df.shape}")

---
## 3. Outlier: rilevamento e decisione

In [ ]:
# Metodi di rilevamento degli outlier

def detectar_outliers_iqr(serie, factor=1.5):
    """Regola IQR: outlier se < Q1 - 1.5·IQR o > Q3 + 1.5·IQR"""
    Q1, Q3 = serie.quantile(0.25), serie.quantile(0.75)
    IQR = Q3 - Q1
    return (serie < Q1 - factor*IQR) | (serie > Q3 + factor*IQR)

def detectar_outliers_zscore(serie, umbral=3.0):
    """Z-score: outlier se |z| > soglia"""
    return (np.abs(stats.zscore(serie.dropna())) > umbral)

cols_num = df.select_dtypes('number').columns.drop('quality')

print(f"{'Colonna':30} {'Outlier IQR':>14} {'% IQR':>8} {'Outlier Z>3':>14}")
print('-' * 70)
for col in cols_num:
    n_iqr = detectar_outliers_iqr(df[col]).sum()
    pct_iqr = n_iqr / len(df) * 100
    n_z   = detectar_outliers_zscore(df[col]).sum()
    print(f"  {col:28} {n_iqr:14} {pct_iqr:8.1f}% {n_z:14}")

# Visualizzare gli outlier con i boxplot
fig, axes = plt.subplots(3, 4, figsize=(15, 9))
for ax, col in zip(axes.ravel(), cols_num):
    ax.boxplot(df[col].dropna(), patch_artist=True,
               boxprops=dict(facecolor='steelblue', alpha=0.6))
    ax.set_title(col, fontsize=9)
    ax.tick_params(labelbottom=False)
    ax.grid(True, alpha=0.3, axis='y')
axes.ravel()[-1].set_visible(False)
plt.suptitle('Outlier per variabile (boxplot IQR)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Decisione sugli outlier: clipping (Winsorizzazione)
# Tagliamo al percentile 1 e 99 per non perdere dati

df_limpio = df.copy()

cols_a_clipear = ['fixed acidity', 'volatile acidity', 'citric acid',
                   'residual sugar', 'chlorides', 'free sulfur dioxide',
                   'total sulfur dioxide', 'sulphates']

limites = {}
for col in cols_a_clipear:
    p1, p99 = df_limpio[col].quantile(0.01), df_limpio[col].quantile(0.99)
    antes = detectar_outliers_iqr(df_limpio[col]).sum()
    df_limpio[col] = df_limpio[col].clip(p1, p99)
    despues = detectar_outliers_iqr(df_limpio[col]).sum()
    limites[col] = (p1, p99)
    print(f"  {col:30}: [{p1:.4f}, {p99:.4f}] — outlier IQR: {antes}→{despues}")

print(f"\n✓ Clipping applicato su {len(cols_a_clipear)} colonne")
print(f"  Le colonne density, pH, alcohol hanno outlier ragionevoli → nessun clipping")

---
## 4. Encoding e trasformazioni

In [ ]:
# Encoding della variabile 'tipo' (rosso/bianco)

# Opzione 1: Label Encoding (per modelli che accettano valori ordinali)
df_limpio['tipo_encoded'] = (df_limpio['tipo'] == 'tinto').astype(int)  # 1=rosso, 0=bianco

# Opzione 2: One-Hot Encoding
df_ohe = pd.get_dummies(df_limpio, columns=['tipo'], prefix='tipo', drop_first=False)
print("Colonne dopo il One-Hot Encoding:")
print([c for c in df_ohe.columns if 'tipo' in c])

# Trasformazioni della distribuzione
# Alcune variabili sono molto asimmetriche → la log transform le normalizza
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
cols_sesgadas = ['residual sugar', 'chlorides', 'free sulfur dioxide']

for i, col in enumerate(cols_sesgadas):
    # Originale
    axes[0, i].hist(df_limpio[col], bins=40, color='coral', edgecolor='white')
    sk = df_limpio[col].skew()
    axes[0, i].set_title(f'{col}\nskewness={sk:.2f}')
    axes[0, i].set_xlabel('Originale')

    # Log-transform
    log_vals = np.log1p(df_limpio[col])  # log(1+x) per evitare log(0)
    sk_log = log_vals.skew()
    axes[1, i].hist(log_vals, bins=40, color='steelblue', edgecolor='white')
    axes[1, i].set_title(f'log1p({col})\nskewness={sk_log:.2f}')
    axes[1, i].set_xlabel('Log-trasformata')

plt.suptitle('Effetto della trasformazione logaritmica', fontsize=13)
plt.tight_layout()
plt.show()

# Applicare la log-transform alle variabili più asimmetriche
for col in cols_sesgadas:
    df_limpio[f'{col}_log'] = np.log1p(df_limpio[col])

print("\nLog-transform aggiunte:", [f'{c}_log' for c in cols_sesgadas])

---
## 5. Feature Engineering e salvataggio finale

In [ ]:
# Feature Engineering: creare variabili derivate con significato enologico

# Rapporto anidride solforosa libera/totale: indica quanto SO2 è 'attivo'
df_limpio['so2_ratio'] = (
    df_limpio['free sulfur dioxide'] / (df_limpio['total sulfur dioxide'] + 1e-8)
)

# Indice di acidità: combina acidità fissa e volatile
df_limpio['acid_index'] = df_limpio['fixed acidity'] - df_limpio['volatile acidity']

# Densità relativa: quanto si discosta la densità da quella dell'acqua (1.0)
df_limpio['density_diff'] = np.abs(df_limpio['density'] - 1.0)

# Variabile obiettivo aggiuntiva: classificazione binaria
# Buon vino = qualità >= 7
df_limpio['buen_vino'] = (df_limpio['quality'] >= 7).astype(int)

print(f"Nuove feature: so2_ratio, acid_index, density_diff")
print(f"Target binario:  buen_vino (1 se quality >= 7)")
print(f"\nDistribuzione di 'buen_vino':")
print(df_limpio['buen_vino'].value_counts())
print(f"  Proporzione buoni: {df_limpio['buen_vino'].mean():.2%}")

# Correlazione delle feature con il target
cols_features = [c for c in df_limpio.select_dtypes('number').columns
                 if c not in ['quality', 'buen_vino', 'tipo_encoded']]
corr_target = df_limpio[cols_features + ['buen_vino']].corr()['buen_vino'].drop('buen_vino')
corr_target = corr_target.sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
colores = ['green' if v > 0 else 'red' for v in corr_target.values]
ax.barh(corr_target.index, corr_target.values, color=colores, edgecolor='white')
ax.axvline(0, color='black', lw=1)
ax.set_xlabel('Correlazione con buen_vino'); ax.set_title('Feature vs target')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

In [ ]:
import json
from datetime import datetime

# Definire le colonne finali per il modello
FEATURES_ML = [
    'fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar',
    'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density',
    'pH', 'sulphates', 'alcohol',
    'so2_ratio', 'acid_index', 'density_diff',
    'tipo_encoded'
]
TARGET_REG = 'quality'
TARGET_CLF = 'buen_vino'

# Dataset finale pulito
df_final = df_limpio[FEATURES_ML + [TARGET_REG, TARGET_CLF, 'tipo']].copy()

print(f"Dataset elaborato: {df_final.shape}")
print(f"Feature: {FEATURES_ML}")
print(f"Target: {TARGET_REG} (regressione), {TARGET_CLF} (classificazione)")

# Salvare
df_final.to_parquet('data/processed/vinos_procesado.parquet', index=False)
df_final.to_csv('data/processed/vinos_procesado.csv', index=False)

# Metadati della fase di preprocessing
meta_proc = {
    'fecha': datetime.now().isoformat(),
    'filas_raw':       int(df_raw.shape[0]),
    'filas_procesado': int(df_final.shape[0]),
    'features':        FEATURES_ML,
    'target_regresion': TARGET_REG,
    'target_clasificacion': TARGET_CLF,
    'limites_clipping': {k: [float(v[0]), float(v[1])] for k,v in limites.items()},
    'proporcion_buenos': float(df_final[TARGET_CLF].mean()),
    'pasos': [
        'Eliminare i duplicati',
        'Clipping p1-p99 sulle variabili asimmetriche',
        'Label encoding di tipo',
        'Log-transform su residual sugar, chlorides, free SO2',
        'Feature engineering: so2_ratio, acid_index, density_diff',
        'Creare il target binario buen_vino (quality >= 7)'
    ]
}
with open('data/processed/preprocesado_meta.json', 'w', encoding='utf-8') as f:
    json.dump(meta_proc, f, indent=2, ensure_ascii=False)

print("\n✓ Dataset elaborato salvato in data/processed/")

---
## Esercitazione pratica

### Esercizio 1 — Mappa delle correlazioni

**Consegna:**
1. Calcola la matrice di correlazione di Pearson per tutte le feature numeriche
2. Rappresentala come heatmap con `seaborn.heatmap`
3. Identifica le 3 coppie di variabili più correlate
4. Qualcuna potrebbe essere eliminata perché ridondante?

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 1</summary>

```python
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

df = pd.read_parquet('data/processed/vinos_procesado.parquet')
cols_num = df.select_dtypes('number').columns.drop(['quality', 'buen_vino'])
corr = df[cols_num].corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr, dtype=bool))  # nascondere il triangolo superiore
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, linewidths=0.5, ax=ax)
ax.set_title('Mappa delle correlazioni — Wine Quality', fontsize=14)
plt.tight_layout()
plt.show()

# Top 3 coppie
corr_abs = corr.abs()
np.fill_diagonal(corr_abs.values, 0)
pares = []
for _ in range(3):
    idx = np.unravel_index(corr_abs.values.argmax(), corr_abs.shape)
    c1, c2 = corr_abs.columns[idx[0]], corr_abs.columns[idx[1]]
    pares.append((c1, c2, corr.loc[c1, c2]))
    corr_abs.loc[c1, c2] = 0
    corr_abs.loc[c2, c1] = 0

print("Top 3 correlazioni:")
for c1, c2, r in pares:
    print(f"  {c1} ↔ {c2}: r={r:.3f}")
```
</details>

---
### Esercizio 2 — Confronto delle distribuzioni per qualità

**Consegna:** In cosa differiscono le feature tra vini di bassa e alta qualità?

1. Per ogni feature numerica, calcola la media per gruppo (buen_vino=0 vs 1)
2. Calcola il t-test tra i gruppi (`scipy.stats.ttest_ind`)
3. Mostra le feature con differenza statisticamente significativa (p < 0.05)
4. Ordinale per dimensione dell'effetto (differenza di medie normalizzata)

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 2</summary>

```python
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

df = pd.read_parquet('data/processed/vinos_procesado.parquet')
features = [c for c in df.select_dtypes('number').columns
            if c not in ['quality', 'buen_vino']]

buenos    = df[df['buen_vino'] == 1]
no_buenos = df[df['buen_vino'] == 0]

resultados = []
for feat in features:
    b  = buenos[feat].dropna()
    nb = no_buenos[feat].dropna()
    t, p = stats.ttest_ind(b, nb)
    efecto = (b.mean() - nb.mean()) / (df[feat].std() + 1e-8)  # Cohen's d semplificato
    resultados.append({'feature': feat, 'media_buenos': b.mean(),
                       'media_no_buenos': nb.mean(), 'p_value': p, 'efecto': efecto})

df_res = pd.DataFrame(resultados).sort_values('efecto', key=abs, ascending=False)
sig = df_res[df_res['p_value'] < 0.05]

print(f"Feature significative (p<0.05): {len(sig)}/{len(features)}")
print(sig[['feature','media_buenos','media_no_buenos','p_value','efecto']].round(4).to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
colores = ['green' if v > 0 else 'red' for v in sig['efecto']]
ax.barh(sig['feature'], sig['efecto'], color=colores, edgecolor='white')
ax.axvline(0, color='black', lw=1)
ax.set_xlabel('Effetto (positivo = più nei buoni, negativo = più nei non buoni)')
ax.set_title('Dimensione dell\'effetto per feature')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout(); plt.show()
```
</details>

---
### Esercizio 3 — Pipeline di preprocessing riproducibile

**Consegna:** Incapsula l'intero preprocessing in una funzione pura:

```python
def preprocesar(df_raw, config) -> pd.DataFrame:
    ...
```

Dove `config` è un dict con i parametri (soglia di qualità, colonne da sottoporre a clipping, ecc.).
La funzione deve essere **idempotente**: chiamarla due volte con lo stesso input dà lo stesso risultato.

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 3</summary>

```python
import pandas as pd
import numpy as np

CONFIG_DEFAULT = {
    'umbral_calidad_buena': 7,
    'cols_clip': ['fixed acidity', 'volatile acidity', 'citric acid',
                  'residual sugar', 'chlorides', 'free sulfur dioxide',
                  'total sulfur dioxide', 'sulphates'],
    'percentil_clip': (0.01, 0.99),
    'features_finales': [
        'fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar',
        'chlorides', 'free sulfur dioxide', 'total sulfur dioxide',
        'density', 'pH', 'sulphates', 'alcohol',
        'so2_ratio', 'acid_index', 'density_diff', 'tipo_encoded'
    ]
}

def preprocesar(df_raw, config=None):
    if config is None:
        config = CONFIG_DEFAULT
    df = df_raw.copy()

    # 1. Eliminare i duplicati
    df = df.drop_duplicates().reset_index(drop=True)

    # 2. Clipping
    p_low, p_high = config['percentil_clip']
    for col in config['cols_clip']:
        if col in df.columns:
            lo, hi = df[col].quantile(p_low), df[col].quantile(p_high)
            df[col] = df[col].clip(lo, hi)

    # 3. Encoding
    df['tipo_encoded'] = (df['tipo'] == 'tinto').astype(int)

    # 4. Feature engineering
    df['so2_ratio']    = df['free sulfur dioxide'] / (df['total sulfur dioxide'] + 1e-8)
    df['acid_index']   = df['fixed acidity'] - df['volatile acidity']
    df['density_diff'] = np.abs(df['density'] - 1.0)

    # 5. Target
    df['buen_vino'] = (df['quality'] >= config['umbral_calidad_buena']).astype(int)

    return df[config['features_finales'] + ['quality', 'buen_vino', 'tipo']]

# Test di idempotenza
df_raw = pd.read_parquet('data/raw/vinos_combinados_raw.parquet')
df_p1 = preprocesar(df_raw)
df_p2 = preprocesar(df_raw)  # seconda esecuzione
assert (df_p1.values == df_p2.values).all(), "Non è idempotente!"
print(f"✓ Funzione idempotente. Risultato: {df_p1.shape}")
```
</details>

---
### Esercizio 4 — Analisi del bias per tipo di vino

**Consegna:** I vini rossi e bianchi hanno la stessa distribuzione di qualità?

1. Confronta la distribuzione di `quality` tra rossi e bianchi
2. Applica un test di Mann-Whitney U (`scipy.stats.mannwhitneyu`) per vedere se sono uguali
3. Bisognerebbe stratificare lo split train/test per tipo di vino?

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 4</summary>

```python
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

df = pd.read_parquet('data/processed/vinos_procesado.parquet')

tintos  = df[df['tipo'] == 'tinto']['quality']
blancos = df[df['tipo'] == 'blanco']['quality']

u_stat, p_value = stats.mannwhitneyu(tintos, blancos, alternative='two-sided')
print(f"Mann-Whitney U: U={u_stat:.0f}, p={p_value:.6f}")
print(f"{'Distribuzioni significativamente diverse' if p_value < 0.05 else 'Nessuna differenza significativa'}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for tipo, color in [('tinto','firebrick'), ('blanco','steelblue')]:
    datos = df[df['tipo']==tipo]['quality']
    datos.value_counts().sort_index().plot(
        kind='bar', ax=axes[0], color=color, alpha=0.6, label=tipo)
    datos.plot(kind='kde', ax=axes[1], color=color, lw=2, label=tipo)

axes[0].set_title('Distribuzione della qualità per tipo'); axes[0].legend()
axes[1].set_title('KDE della qualità'); axes[1].legend()
for ax in axes: ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print(f"\nMedia qualità — Rosso: {tintos.mean():.2f}, Bianco: {blancos.mean():.2f}")
print(f"% buoni — Rosso: {(tintos>=7).mean():.2%}, Bianco: {(blancos>=7).mean():.2%}")
print("\n→ Con differenze significative, usare stratify=['buen_vino','tipo'] in train_test_split")
```
</details>

---
### Esercizio 5 (sfida) — Report HTML di qualità dei dati

**Consegna:** Genera un report HTML automatico del dataset elaborato:

1. Installa `ydata-profiling` (`pip install ydata-profiling`) oppure usa pandas + matplotlib
2. Se hai ydata-profiling: `ProfileReport(df).to_file('docs/reporte_calidad.html')`
3. Altrimenti: crea una funzione che generi un HTML con tabella di statistiche + immagini base64
4. Il report deve includere: shape, nulli, distribuzioni, correlazioni

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 5</summary>

```python
import pandas as pd
import matplotlib.pyplot as plt
import base64, io
from pathlib import Path

def fig_a_base64(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format='png', bbox_inches='tight', dpi=100)
    plt.close(fig)
    return base64.b64encode(buf.getvalue()).decode()

def generar_reporte_html(df, ruta_salida='docs/reporte_calidad.html'):
    Path('docs').mkdir(exist_ok=True)
    imagenes = []

    # Istogrammi
    cols_num = df.select_dtypes('number').columns.drop(['quality','buen_vino']).tolist()
    fig, axes = plt.subplots(3, 5, figsize=(18, 10))
    for ax, col in zip(axes.ravel(), cols_num):
        ax.hist(df[col], bins=30, color='steelblue', edgecolor='white')
        ax.set_title(col, fontsize=8); ax.grid(True, alpha=0.3)
    for ax in axes.ravel()[len(cols_num):]:
        ax.set_visible(False)
    fig.suptitle('Distribuzioni')
    imagenes.append(('Distribuzioni', fig_a_base64(fig)))

    # Statistiche
    tabla_html = df.describe().round(3).to_html(classes='tabla')

    html = f"""
    <!DOCTYPE html><html lang="it"><head>
    <meta charset="UTF-8"><title>Report di Qualità</title>
    <style>body{{font-family:Arial;margin:30px}} .tabla{{border-collapse:collapse;width:100%}}
    .tabla td,.tabla th{{border:1px solid #ddd;padding:6px;font-size:12px}}
    .tabla th{{background:#4472C4;color:white}} h1,h2{{color:#333}}</style></head>
    <body><h1>Report di Qualità — Wine Quality Dataset</h1>
    <p>Righe: {df.shape[0]:,} | Colonne: {df.shape[1]} | Nulli: {df.isna().sum().sum()}</p>
    <h2>Statistiche</h2>{tabla_html}
    {''.join(f"<h2>{t}</h2><img src='data:image/png;base64,{img}' style='max-width:100%'>" for t, img in imagenes)}
    </body></html>
    """

    Path(ruta_salida).write_text(html, encoding='utf-8')
    print(f"Report generato: {ruta_salida}")

df = pd.read_parquet('data/processed/vinos_procesado.parquet')
generar_reporte_html(df)
```
</details>

---
## Riepilogo

```python
# Pipeline di pulizia
df = df_raw.copy()
df = df.drop_duplicates().reset_index(drop=True)          # duplicati
df['col'] = df['col'].fillna(df['col'].median())           # nulli
df['col'] = df['col'].clip(df['col'].quantile(0.01),       # outlier
                            df['col'].quantile(0.99))

# Encoding
df['cat_encoded'] = (df['cat'] == 'valor').astype(int)     # binario
df = pd.get_dummies(df, columns=['cat'])                    # one-hot

# Trasformazioni
df['col_log'] = np.log1p(df['col'])                        # log-transform

# Feature engineering
df['ratio'] = df['a'] / (df['b'] + 1e-8)
df['diff']  = df['a'] - df['b']

# Target binario
df['target_bin'] = (df['score'] >= umbral).astype(int)

# Salvare
df.to_parquet('data/processed/procesado.parquet', index=False)
```

## Cosa vedremo nella prossima lezione?
Nella **Lezione 3** archivieremo il dataset in un database **SQL** (SQLite), impareremo a fare query e useremo SQLAlchemy per collegare Python al database.